# Baseline model that runs on videos

In [ ]:
import ipywidgets.widgets as widgets
from IPython.display import display, HTML
import sys
import cv2
import numpy as np
import threading
import traitlets
from traitlets.config.configurable import SingletonConfigurable
import time

# Create widgets for the displaying of the images
display_color = widgets.Image(format='jpeg', width='50%')
display_processed = widgets.Image(format='jpeg', width='50%') 
layout = widgets.Layout(width='100%')
sidebyside = widgets.HBox([display_color, display_processed], layout=layout) 
display(sidebyside) 

def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg', value)[1])

# VideoReader class to capture frames from the video file in a separate thread
class VideoReader(SingletonConfigurable):
    color_value = traitlets.Any() 
    
    def __init__(self):
        super(VideoReader, self).__init__()
        self.cap = cv2.VideoCapture('baseline anticlockwise lap (52s).mp4')
        if not self.cap.isOpened():
            print("Error: Could not open video file")
        self.thread_running_flag = False

    def _capture_frames(self):
        while(self.thread_running_flag == True):
            ret, frame = self.cap.read()
            if not ret:
                self.cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                continue
                
            self.color_value = frame
            time.sleep(0.03)
                
    def start(self): 
        if self.thread_running_flag == False: 
            self.thread_running_flag = True 
            self.thread = threading.Thread(target=self._capture_frames) 
            self.thread.start() 

    def stop(self): 
        if self.thread_running_flag == True:
            self.thread_running_flag = False 
            self.thread.join() 
            self.cap.release()

# Global variable to remember the last turn direction
last_turn_action = None

# Callback function, invoked when traitlets detects the changing of the color image
def process_vision(change):
    global last_turn_action # Access the global state variable
    
    frame = change['new'].copy()
    height, width, _ = frame.shape
    
    # Focus on the centre of the image
    top_crop = int(height * 0.25)
    bottom_crop = int(height * 0.75)
    left_crop = int(width * 0.25)
    right_crop = int(width * 0.75)
    
    # Black out the areas outside the active region
    frame[:top_crop, :] = (0, 0, 0)
    frame[bottom_crop:, :] = (0, 0, 0)
    frame[:, :left_crop] = (0, 0, 0)
    frame[:, right_crop:] = (0, 0, 0)
    
    # HSV color space conversion and thresholding to isolate yellow pixels
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV) 
    mask = cv2.inRange(hsv, (10, 60, 60), (45, 255, 255))
    #mask = cv2.inRange(hsv, (14, 30, 30), (45, 255, 255)) 

    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel) 
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel) 

    # Define thickness of the check zones at the bottom of the image
    zone_thickness = 20
    
    # Calculate the total active width and divide by 3
    active_width = right_crop - left_crop
    segment_width = active_width // 3
    
    # Define the x-coordinates for the divisions
    x1 = left_crop
    x2 = left_crop + segment_width
    x3 = left_crop + (2 * segment_width)
    x4 = right_crop 
    
    # Isolate the mask arrays for the three equal bottom sections
    bottom_left_zone   = mask[bottom_crop - zone_thickness:bottom_crop, x1:x2]
    bottom_center_zone = mask[bottom_crop - zone_thickness*3:bottom_crop, x2:x3]
    bottom_right_zone  = mask[bottom_crop - zone_thickness:bottom_crop, x3:x4]
    
    # If there are 20 or more yellow pixels in a zone, trigger that action
    pixel_threshold = 20 * 255 
    
    touching_bottom_left   = np.sum(bottom_left_zone) > pixel_threshold
    touching_bottom_center = np.sum(bottom_center_zone) > pixel_threshold
    touching_bottom_right  = np.sum(bottom_right_zone) > pixel_threshold

    #Hypothetical motor control logic based on the zones detected
    if touching_bottom_left: 
        last_turn_action = 'left' # Remember the turn
        print('\rAction: TURN LEFT (motors.left(0.5))    ', end='', flush=True)
        cv2.putText(frame, "TURN LEFT", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    elif touching_bottom_right:
        last_turn_action = 'right' # Remember the turn
        print('\rAction: TURN RIGHT (motors.right(0.5))  ', end='', flush=True)
        cv2.putText(frame, "TURN RIGHT", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    elif touching_bottom_center:
        print('\rAction: FORWARD (motors.forward(0.1))   ', end='', flush=True)
        cv2.putText(frame, "FORWARD", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    else:
        # Fallback to the last known turn if the line is lost
        if last_turn_action == 'left':
            print('\rAction: LOST LINE - TURN LEFT     ', end='', flush=True)
            cv2.putText(frame, "LOST LINE - TURN LEFT", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 165, 255), 2)
        elif last_turn_action == 'right':
            print('\rAction: LOST LINE - TURN RIGHT    ', end='', flush=True)
            cv2.putText(frame, "LOST LINE - TURN RIGHT", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 165, 255), 2)
        else:
            print('\rAction: STOP (motors.stop())            ', end='', flush=True)
            cv2.putText(frame, "STOP - NO LINE DATA", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # Draw the 3 zones on the camera feed
    cv2.rectangle(frame, (x1, bottom_crop - zone_thickness), (x2, bottom_crop), (255, 0, 0), 2)
    cv2.rectangle(frame, (x2, bottom_crop - zone_thickness*3), (x3, bottom_crop), (0, 255, 0), 2)
    cv2.rectangle(frame, (x3, bottom_crop - zone_thickness), (x4, bottom_crop), (0, 0, 255), 2)

    # Update the display widgets with the resized images
    scale = 0.5 
    resized_color = cv2.resize(frame, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    resized_mask = cv2.resize(mask, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    
    display_color.value = bgr8_to_jpeg(resized_color)
    display_processed.value = bgr8_to_jpeg(resized_mask)

# Create the video reader object and attach observer
reader = VideoReader()
reader.observe(process_vision, names=['color_value'])
reader.start()

Action: FORWARD (motors.forward(0.1))   